# Algoritmo de Bellman-Ford

Calcula distancias mínimas desde un origen admitiendo pesos **negativos**. Relaja todas las aristas $V - 1$ veces (ningún camino mínimo tiene más aristas); si en una pasada más todavía mejora alguna distancia, hay un ciclo de peso negativo. Cuesta $O(V \cdot E)$.

## 1. Distancias con pesos negativos

Calcular las distancias mínimas desde `A` en un grafo dado como lista de aristas `(origen, destino, peso)`, detectando ciclos negativos.

In [1]:
def bellman_ford(nodos, aristas, origen):
    dist = {n: float("inf") for n in nodos}
    dist[origen] = 0
    for _ in range(len(nodos) - 1):
        cambio = False
        for u, v, w in aristas:
            if dist[u] + w < dist[v]:
                dist[v] = dist[u] + w
                cambio = True
        # sin cambios: ya es definitivo
        if not cambio:
            break
    if any(dist[u] + w < dist[v] for u, v, w in aristas):
        return None
    return dist

nodos = "ABCDE"
aristas = [("A", "B", 6), ("A", "C", 7), ("B", "D", 5), ("B", "E", -4), ("C", "D", -3), ("C", "E", 9), ("D", "B", -2), ("E", "D", 7)]
print(bellman_ford(nodos, aristas, "A"))

{'A': 0, 'B': 2, 'C': 7, 'D': 4, 'E': -2}


## 2. Arbitraje de divisas

Con tipos de cambio $r_{ij}$, existe arbitraje si cambiando en ciclo se acaba con más dinero: $\prod r > 1 \iff \sum -\log r < 0$. Detectarlo buscando ciclos negativos con pesos $-\log r$.

In [2]:
from math import log

cambios = {
    ("EUR", "USD"): 1.08, ("USD", "EUR"): 0.93,
    ("USD", "JPY"): 150.0, ("JPY", "USD"): 0.0067,
    ("EUR", "JPY"): 158.0, ("JPY", "EUR"): 0.0064,
}

def hay_arbitraje(cambios):
    divisas = {d for par in cambios for d in par}
    aristas = [(a, b, -log(r)) for (a, b), r in cambios.items()]
    # origen ficticio conectado a todo para ver todos los ciclos
    aristas += [("*", d, 0) for d in divisas]
    return bellman_ford(divisas | {"*"}, aristas, "*") is None

print(hay_arbitraje(cambios))
print(1.08 * 150.0 * 0.0064)

True
1.0368
